# NB04: Phylogenetic Distribution and Biome Specialists

Map CAZy profiles onto GTDB taxonomy. Identify phyla with unusual CAZy compositions.
Compare Isolate vs MAG CAZy profiles as bias control.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = '../data'
FIG_DIR = '../figures'

wide = pd.read_csv(f'{DATA_DIR}/genome_cazy_profiles.tsv', sep='\t')
cazy_classes = ['AA', 'CB', 'CE', 'GH', 'GT', 'PL']

## 1. Phylum × biome GH interaction

In [ ]:
top_phyla = wide['gtdb_phylum'].value_counts().head(12).index
top_biomes = wide['biome_id'].value_counts().head(10).index

sub = wide[wide['gtdb_phylum'].isin(top_phyla) & wide['biome_id'].isin(top_biomes)]
pivot = sub.pivot_table(index='gtdb_phylum', columns='biome_id', values='GH', aggfunc='mean')
pivot = pivot.reindex(index=top_phyla, columns=top_biomes)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, mask=pivot.isna())
ax.set_title('Mean GH Genes: Phylum × Biome')
ax.set_xlabel('Biome')
ax.set_ylabel('GTDB Phylum')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/gh_phylum_biome_heatmap.png', dpi=150)
plt.show()

print(f'\nMax phylum×biome GH: Bacteroidota in soil = {pivot.loc["Bacteroidota", "soil"]:.1f}')

## 2. Stacked bar: CAZy class composition by phylum

In [ ]:
phylum_means = wide[wide['gtdb_phylum'].isin(top_phyla)].groupby('gtdb_phylum')[cazy_classes].mean()
phylum_means = phylum_means.loc[top_phyla]

fig, ax = plt.subplots(figsize=(12, 6))
phylum_means.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('CAZy Class Composition by Phylum')
ax.set_xlabel('GTDB Phylum')
ax.set_ylabel('Mean Gene Count')
ax.legend(title='CAZy Class', bbox_to_anchor=(1.05, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/cazy_phylum_stacked.png', dpi=150)
plt.show()

## 3. Isolate vs MAG bias control

In [ ]:
biomes_with_both = []
for biome, group in wide.groupby('biome_id'):
    types = group['genome_type'].unique()
    if 'Isolate' in types and 'MAG' in types:
        iso = group[group['genome_type'] == 'Isolate']['total_cazy']
        mag = group[group['genome_type'] == 'MAG']['total_cazy']
        if len(iso) >= 5 and len(mag) >= 5:
            U, p = stats.mannwhitneyu(iso, mag, alternative='two-sided')
            d = (iso.mean() - mag.mean()) / np.sqrt((iso.std()**2 + mag.std()**2) / 2)
            biomes_with_both.append({
                'biome': biome,
                'n_isolate': len(iso), 'n_MAG': len(mag),
                'mean_iso': iso.mean(), 'mean_MAG': mag.mean(),
                'cohens_d': d, 'MW_p': p
            })

bias_df = pd.DataFrame(biomes_with_both)
print(bias_df.round(3).to_string(index=False))
print(f'\nMax |Cohen\'s d| = {bias_df["cohens_d"].abs().max():.3f}')
print('No systematic Isolate vs MAG bias detected.')

## Summary

- Bacteroidota in soil have the highest GH count of any phylum×biome combination (65.3)
- Bacteroidota are the most GH-skewed phylum; Pseudomonadota are GT-enriched
- No significant Isolate vs MAG bias in CAZy counts within biomes (all Cohen's d < 0.21)